# Contexta Research — 02 Corpus Preprocessing

Built on SenseForge shared modules. Run sequentially from the repository root or notebooks directory. Missing data is explicitly skipped; no fabricated results.

In [2]:
from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

DATA_ROOT = Path("/Users/suvratsmacbookair/MyLocalFiles/NLP project")
os.environ["CONTEXTA_DATA_ROOT"] = str(DATA_ROOT)
os.environ["SENSEFORGE_DATA_ROOT"] = str(DATA_ROOT)

ART = ROOT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Data:", DATA_ROOT)


Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Data: /Users/suvratsmacbookair/MyLocalFiles/NLP project


## Prepare full held-out hash protection

In [4]:
from senseforge.data import source_paths
from senseforge.research import protected_hashes,iter_clean_pairs,write_jsonl,digest
paths=source_paths();held=[paths[k] for k in ('iit_validation','iit_test') if paths[k].exists()]
forbidden=protected_hashes(held) if len(held)==2 else None
print('Held-out splits complete:',forbidden is not None)
os.environ["CONTEXTA_SAMANANTAR_DIRECTION"] = "en-hi"

Held-out splits complete: True


## Stream corpus and reject held-out duplicates

In [6]:
import json
if forbidden is None:print('SKIPPED: both official held-out splits required')
else:
    output=ART/'clean_parallel.jsonl';seen=set();n=0
    with output.open('w',encoding='utf8') as f:
        for name,path in paths.items():
            if name!='iit_train' and not name.startswith('samanantar_'):continue
            if not path.exists():continue
            direction='en-hi' if name=='iit_train' else os.environ.get('CONTEXTA_SAMANANTAR_DIRECTION')
            if direction not in ('en-hi','hi-en'):
                print('SKIPPED ambiguous direction:',name);continue
            for en,hi in iter_clean_pairs(path,forbidden,direction):
                key=digest(en+'|'+hi)
                if key in seen:continue
                seen.add(key);f.write(json.dumps({'en':en,'hi':hi,'source':name,'split':'train'},ensure_ascii=False)+'\n');n+=1
    print('Wrote',n,'training-only pairs to',output)

Wrote 6019563 training-only pairs to /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/clean_parallel.jsonl


In [7]:
import pyarrow.parquet as pq

path = DATA_ROOT / "Dataset_Samanantar" / "train-00000-of-00008.parquet"
batch = next(pq.ParquetFile(path).iter_batches(batch_size=3))

for row in batch.to_pylist():
    print("SRC:", str(row.get("src", ""))[:200])
    print("TGT:", str(row.get("tgt", ""))[:200])
    print("---")

SRC: However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles
TGT: आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।
---
SRC: Whosoever desires the reward of the world, with Allah is the reward of the world and of the Everlasting Life. Allah is the Hearer, the Seer.
TGT: और जो शख्स (अपने आमाल का) बदला दुनिया ही में चाहता है तो ख़ुदा के पास दुनिया व आख़िरत दोनों का अज्र मौजूद है और ख़ुदा तो हर शख्स की सुनता और सबको देखता है
---
SRC: The value of insects in the biosphere is enormous because they outnumber all other living groups in measure of species richness.
TGT: जैव-मंडल में कीड़ों का मूल्य बहुत है, क्योंकि प्रजातियों की समृद्धि के मामले में उनकी संख्या अन्य जीव समूहों से ज़्यादा है।
---
